# Vision SFT: receipt image → structured JSON (CORD)

Fine-tune a small **vision-language** model to read a receipt **image** and emit structured JSON (line items + totals). Real document images, human gold labels — no synthetic inputs.

**Why this is a good showcase:** document→JSON extraction is one of the most common enterprise VLM use cases, the metric is a hard field-level accuracy number (not vibes), and the dataset ([`naver-clova-ix/cord-v2`](https://huggingface.co/datasets/naver-clova-ix/cord-v2), 1K receipts) is small enough to run cheaply.

**The recipe:** prepare data → measure the **base** model → LoRA SFT → measure the **tuned** model → compare. This is step 1 of a trilogy; step 2 (distillation to a larger unlabeled receipt pool) and step 3 (DPO to abstain on illegible fields) build on the model this produces.

**Prereqs:** Jupyter kernel = conda `cookbook` env, `FIREWORKS_API_KEY` in `training/.env`. Vision SFT is supported by the same `train_sft.py` used in `examples/sft/run_vl.sh`.

In [ ]:
# Run once if imports fail.
import sys
!{sys.executable} -m pip install -q -e "../../../.[eval]" datasets

In [ ]:
# --- edit these ---
BASE_MODEL = "accounts/fireworks/models/qwen3-vl-8b-instruct"
TOKENIZER_MODEL = "Qwen/Qwen3-VL-8B-Instruct"

# After training + deploying the LoRA, set this to the deployed model id to eval it.
# e.g. "accounts/<your-account>/models/sft-cord-..."  (leave None to only eval the base model)
TUNED_MODEL = None

TRAIN_MAX = 400      # CORD train is 800; cap for a faster/cheaper run
EVAL_MAX = 50        # rows from the test split to score
EPOCHS = 3
LORA_RANK = 16
CONCURRENCY = 4

In [ ]:
import asyncio
import json
import os
import subprocess
import sys
from pathlib import Path

import litellm
from dotenv import load_dotenv

HERE = Path.cwd()
training_dir = next(
    (p for p in [HERE, *HERE.parents] if p.name == "training" and (p / "pyproject.toml").exists()),
    Path("../../../").resolve(),
)
load_dotenv(training_dir / ".env")
if not os.getenv("FIREWORKS_API_KEY"):
    raise EnvironmentError(f"Set FIREWORKS_API_KEY in {training_dir / '.env'}")
litellm.drop_params = True

## 1. Prepare the data

`prepare_cord_sft.py` downloads CORD-v2, flattens each receipt's cryptic `gt_parse` into a clean schema (`items[]`, `subtotal`, `tax`, `service`, `total`; missing → `null`), base64-encodes the image, and writes OpenAI multimodal chat rows. We build a train split (for SFT) and a test split (for eval).

In [ ]:
def prep(split, max_examples, out):
    if Path(out).exists():
        print(f"{out} exists, skipping")
        return out
    cmd = [sys.executable, "prepare_cord_sft.py", "--split", split,
           "--max-examples", str(max_examples), "--out", out]
    subprocess.run(cmd, check=True)
    return out

TRAIN_JSONL = prep("train", TRAIN_MAX, "cord_train.jsonl")
TEST_JSONL = prep("test", EVAL_MAX, "cord_test.jsonl")

test_rows = [json.loads(l) for l in open(TEST_JSONL)]
print(f"train={sum(1 for _ in open(TRAIN_JSONL))}  test={len(test_rows)}")
# peek at one gold target
print("\nsample gold:", test_rows[0]["messages"][1]["content"][:300])

## 2. Scoring

Field-level accuracy against the gold JSON: each of `subtotal/tax/service/total` is an exact-match check, and line items are scored by F1 on `(name, price)` pairs. We average into one `field_score` per receipt so base-vs-tuned is a single comparable number.

In [ ]:
SCALARS = ["subtotal", "tax", "service", "total"]


def parse_json(text):
    """Pull the first JSON object out of a model response."""
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        pass
    i, j = text.find("{"), text.rfind("}")
    if i != -1 and j != -1 and j > i:
        try:
            return json.loads(text[i : j + 1])
        except Exception:
            return None
    return None


def _norm(v):
    return str(v).strip().lower() if v is not None else None


def _item_keys(obj):
    out = set()
    for it in (obj or {}).get("items", []) or []:
        if isinstance(it, dict):
            out.add((_norm(it.get("name")), _norm(it.get("price"))))
    return out


def field_score(pred, gold):
    """Mean of scalar exact-match and item F1. Unparseable pred => 0."""
    if not isinstance(pred, dict):
        return 0.0
    parts = []
    for k in SCALARS:
        parts.append(1.0 if _norm(pred.get(k)) == _norm(gold.get(k)) else 0.0)
    p, g = _item_keys(pred), _item_keys(gold)
    if g or p:
        tp = len(p & g)
        prec = tp / len(p) if p else 0.0
        rec = tp / len(g) if g else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        parts.append(f1)
    return sum(parts) / len(parts)

## 3. Baseline: the un-tuned model

Run the base VLM on the test receipts and score it. This is the number SFT has to beat.

In [ ]:
async def grade_one(model, row, sem):
    gold = json.loads(row["messages"][1]["content"])
    async with sem:
        try:
            resp = await litellm.acompletion(
                model=f"fireworks_ai/{model}" if not model.startswith("fireworks_ai/") else model,
                messages=[row["messages"][0]],  # user turn (image + instruction) only
                temperature=0.0,
                max_tokens=2048,
            )
            pred = parse_json(resp.choices[0].message.content)
        except Exception as e:
            return {"score": 0.0, "error": f"{type(e).__name__}: {str(e)[:80]}"}
    return {"score": field_score(pred, gold), "error": None}


async def evaluate(model, rows):
    sem = asyncio.Semaphore(CONCURRENCY)
    results = await asyncio.gather(*[grade_one(model, r, sem) for r in rows])
    scores = [r["score"] for r in results]
    errs = sum(1 for r in results if r["error"])
    avg = sum(scores) / len(scores) if scores else 0.0
    return avg, errs, results


base_avg, base_errs, _ = await evaluate(BASE_MODEL, test_rows)
print(f"BASE {BASE_MODEL}")
print(f"  field_score = {base_avg:.1%} over {len(test_rows)} receipts ({base_errs} errors)")

## 4. SFT (LoRA) on the receipts

Uses the same `sft_loop` recipe as `examples/sft/run_vl.sh`, just pointed at our CORD JSONL. LoRA keeps it cheap. This provisions a trainer and runs — expect minutes, and it costs GPU time on your account.

After it finishes, **deploy the resulting LoRA** and set `TUNED_MODEL` in the config cell to its model id, then run the eval cell below.

In [ ]:
import training.recipes.sft_loop as sft_loop
from training.utils import TrainerConfig, WandBConfig

OUTPUT_MODEL_ID = "sft-cord-qwen3vl-8b"

config = sft_loop.Config(
    log_path="./cord_sft_logs",
    base_model=BASE_MODEL,
    dataset=str(Path(TRAIN_JSONL).resolve()),
    tokenizer_model=TOKENIZER_MODEL,
    learning_rate=1e-5,
    epochs=EPOCHS,
    batch_size=4,
    max_examples=TRAIN_MAX,
    lora_rank=LORA_RANK,
    output_model_id=OUTPUT_MODEL_ID,
    trainer=TrainerConfig(training_shape_id=""),  # auto-select a validated shape
    wandb=WandBConfig(project="sft-cord", run_name="cord-qwen3vl-8b"),
)
metrics = sft_loop.main(config)
print("SFT complete:", metrics)
print(f"\nNow deploy the LoRA '{OUTPUT_MODEL_ID}' and set TUNED_MODEL to its model id, then run the next cell.")

## 5. Tuned vs base

Set `TUNED_MODEL` (config cell) to your deployed LoRA id and run this to see the lift.

In [ ]:
if TUNED_MODEL:
    tuned_avg, tuned_errs, _ = await evaluate(TUNED_MODEL, test_rows)
    print(f"BASE  {BASE_MODEL}: {base_avg:.1%}")
    print(f"TUNED {TUNED_MODEL}: {tuned_avg:.1%}  ({tuned_errs} errors)")
    print(f"\nLift: {tuned_avg - base_avg:+.1%}")
else:
    print("Set TUNED_MODEL in the config cell (after training + deploying) to run this comparison.")

## Next steps (the rest of the trilogy)

- **Distillation:** CORD is only 1K receipts. Have a stronger teacher VLM label a large *unlabeled* receipt/invoice pool, then SFT on that to generalize beyond CORD's distribution.
- **DPO:** build preference pairs where *chosen* = correct value or `null` for an illegible field, *rejected* = a hallucinated value — to push the model to abstain instead of inventing totals.
- **Scale the schema:** add `store`, `date`, `payment` fields (present in CORD's richer labels) once the basic pipeline works.